# Module 08 — Bytes as Polynomials
## Computational Notebook

**Mathematics of Cryptography · Volume 1**

---

This notebook translates the representation ideas from the tutorial into Python.
You will implement functions that convert bytes to polynomials and back,
verify the round-trip property on all 256 possible byte values,
and get a first computational glimpse at why XOR will become polynomial addition in Module 09.

**No GF(2⁸) multiplication yet.** This notebook is about representation only.

### Prerequisites
- Module 08 tutorial (bytes as polynomials)
- Python 3 (no external libraries needed)

### How to use this notebook
- Read each markdown cell, then run the code cell below it.
- Cells marked **TODO** require you to fill in missing code.
- Cells marked **Run** are complete — just run them to see the output.

---
## Section 1 — Setup

Run this cell first. It defines nothing yet — it just confirms Python is working
and shows you how Python represents a byte as an integer.

In [ ]:
# Run — nothing to change here

# Python already stores byte values as plain integers.
# 0x57 in hex is the same integer as 87 in decimal.
print(f"0x57 as decimal:  {0x57}")
print(f"0x57 as binary:   {0x57:08b}")
print(f"0x83 as decimal:  {0x83}")
print(f"0x83 as binary:   {0x83:08b}")
print()
print("Bit extraction example:")
b = 0x57
for k in range(7, -1, -1):
    bit = (b >> k) & 1
    print(f"  bit position {k}: {bit}  (x^{k} is {'included' if bit else 'absent'})")

---
## Section 2 — Active bit positions

The polynomial terms are exactly the bit positions where the byte has a 1.

**Exercise 2.1** — Implement `active_positions(b)`, which returns a list of bit positions
where `b` has a 1 bit, in **descending order** (highest power first).

For example:
```
active_positions(0x57)  # 01010111  →  [6, 4, 2, 1, 0]
active_positions(0x83)  # 10000011  →  [7, 1, 0]
active_positions(0x00)  # 00000000  →  []
active_positions(0xFF)  # 11111111  →  [7, 6, 5, 4, 3, 2, 1, 0]
```

In [ ]:
def active_positions(b):
    """Return list of bit positions (7..0) where b has a 1 bit, highest first."""
    # TODO: implement this
    # Hint: loop k from 7 down to 0; check (b >> k) & 1
    pass


# Test your implementation
tests = [
    (0x57, [6, 4, 2, 1, 0]),
    (0x83, [7, 1, 0]),
    (0x1B, [4, 3, 1, 0]),
    (0x00, []),
    (0xFF, [7, 6, 5, 4, 3, 2, 1, 0]),
]

print("active_positions tests:")
all_pass = True
for byte_val, expected in tests:
    result = active_positions(byte_val)
    status = "PASS" if result == expected else "FAIL"
    if status == "FAIL":
        all_pass = False
    print(f"  0x{byte_val:02X}  → {result}  (expected {expected})  [{status}]")

print()
print("All tests passed!" if all_pass else "Some tests failed — check your implementation.")

---
## Section 3 — Polynomial string representation

Now build a function that produces the polynomial as a readable string.

**Rules for the output string:**
- Terms are separated by `" + "`
- Position 0 → `"1"` (not `"x^0"`)
- Position 1 → `"x"` (not `"x^1"`)
- Position k ≥ 2 → `"x^k"`
- The zero polynomial (`0x00`) → `"0"`

**Exercise 3.1** — Implement `byte_to_poly(b)` using your `active_positions` function.

In [ ]:
def term_str(k):
    """Return the string for the x^k term."""
    if k == 0:
        return "1"
    elif k == 1:
        return "x"
    else:
        return f"x^{k}"


def byte_to_poly(b):
    """Return the polynomial string for byte value b (0–255)."""
    # TODO: implement this
    # Hint: use active_positions(b) and term_str(k)
    # Don't forget the edge case when b == 0
    pass


# Test your implementation
test_cases = [
    (0x57, "x^6 + x^4 + x^2 + x + 1"),
    (0x83, "x^7 + x + 1"),
    (0x1B, "x^4 + x^3 + x + 1"),
    (0xA5, "x^7 + x^5 + x^2 + 1"),
    (0x63, "x^6 + x^5 + x + 1"),
    (0x00, "0"),
    (0xFF, "x^7 + x^6 + x^5 + x^4 + x^3 + x^2 + x + 1"),
    (0x02, "x"),
    (0x01, "1"),
]

print("byte_to_poly tests:")
all_pass = True
for byte_val, expected in test_cases:
    result = byte_to_poly(byte_val)
    status = "PASS" if result == expected else "FAIL"
    if status == "FAIL":
        all_pass = False
    print(f"  0x{byte_val:02X}  →  {result}")
    if status == "FAIL":
        print(f"         expected: {expected}  [FAIL]")

print()
print("All tests passed!" if all_pass else "Some tests failed.")

---
## Section 4 — Polynomial back to byte

Going the other direction: given a list of active term positions, reconstruct the byte.

**Exercise 4.1** — Implement `poly_to_byte(positions)`, which takes a list of bit positions
(like `[6, 4, 2, 1, 0]`) and returns the corresponding integer byte value.

For example:
```
poly_to_byte([6, 4, 2, 1, 0])  →  0x57  (87)
poly_to_byte([7, 1, 0])        →  0x83  (131)
poly_to_byte([])               →  0x00  (0)
```

In [ ]:
def poly_to_byte(positions):
    """Given a list of active bit positions, return the integer byte value."""
    # TODO: implement this
    # Hint: set bit k by OR-ing with (1 << k)
    pass


# Test
print("poly_to_byte tests:")
tests = [
    ([6, 4, 2, 1, 0], 0x57),
    ([7, 1, 0],       0x83),
    ([4, 3, 1, 0],    0x1B),
    ([],              0x00),
    ([7,6,5,4,3,2,1,0], 0xFF),
]
all_pass = True
for positions, expected in tests:
    result = poly_to_byte(positions)
    status = "PASS" if result == expected else "FAIL"
    if status == "FAIL":
        all_pass = False
    print(f"  {positions}  →  0x{result:02X}  (expected 0x{expected:02X})  [{status}]")

print()
print("All tests passed!" if all_pass else "Some tests failed.")

---
## Section 5 — Round-trip verification

A correct representation must round-trip: converting a byte to its active positions
and back must always recover the original byte.

**Exercise 5.1** — Verify the round-trip for **all 256** possible byte values.

This is a one-liner once your functions are correct.

In [ ]:
# Run — verifies all 256 bytes round-trip correctly

failures = []
for b in range(256):
    recovered = poly_to_byte(active_positions(b))
    if recovered != b:
        failures.append((b, recovered))

if not failures:
    print("Round-trip verified for all 256 byte values.")
    print("Every byte can be converted to its polynomial representation and back.")
else:
    print(f"{len(failures)} failures:")
    for original, got in failures[:10]:
        print(f"  byte 0x{original:02X} → positions → 0x{got:02X}  (mismatch)")

---
## Section 6 — The three-lenses table

The tutorial showed that the same byte can be viewed as hex, binary, an integer, or a polynomial.
Let's print that table for a range of interesting values.

**Run** this cell — no changes needed. Study the output and check that it matches
your expectations from the tutorial.

In [ ]:
# Run

interesting = [0x00, 0x01, 0x02, 0x03, 0x1B, 0x57, 0x63, 0x83, 0xA5, 0xFF]

print(f"{'Hex':<8} {'Binary':<12} {'Decimal':<10} {'Polynomial'}")
print("-" * 70)
for b in interesting:
    print(f"0x{b:02X}     {b:08b}    {b:<10} {byte_to_poly(b)}")

---
## Section 7 — XOR as coefficient addition (preview of Module 09)

The tutorial noted that adding polynomials over GF(2) is the same as XOR.
Here is the computational demonstration — no formal proof yet, just observation.

When we XOR two bytes, the result has a 1 bit in every position where exactly one
of the two input bytes had a 1.  That is exactly what happens when you add the
polynomials term by term modulo 2:

- coefficient 0 + 0 = 0  (XOR: 0 ⊕ 0 = 0)
- coefficient 0 + 1 = 1  (XOR: 0 ⊕ 1 = 1)
- coefficient 1 + 0 = 1  (XOR: 1 ⊕ 0 = 1)
- coefficient 1 + 1 = 0  (XOR: 1 ⊕ 1 = 0)  ← the key one: no "carry"

**Exercise 7.1** — For the pair `0x57` and `0x83`, verify this manually, then
let the code confirm it for all 256×256 pairs.

In [ ]:
# Part A: manual example
a, b = 0x57, 0x83

print(f"a = 0x{a:02X}  =  {byte_to_poly(a)}")
print(f"b = 0x{b:02X}  =  {byte_to_poly(b)}")
print()
xor_result = a ^ b
print(f"a XOR b = 0x{xor_result:02X}  =  {byte_to_poly(xor_result)}")
print()
print("Observe: the polynomial for (a XOR b) has a term x^k exactly when")
print("exactly one of a, b had that term — coefficient addition mod 2.")

In [ ]:
# Part B: verify for all 256 × 256 pairs
# (This takes a moment — there are 65,536 pairs to check)

def poly_add_mod2(a, b):
    """Add two byte-polynomials over GF(2): XOR the bit patterns."""
    return a ^ b

mismatches = 0
for a in range(256):
    for b in range(256):
        xor_way  = a ^ b
        poly_way = poly_add_mod2(a, b)
        if xor_way != poly_way:
            mismatches += 1

print(f"Checked all 65,536 pairs.")
print(f"Mismatches between XOR and polynomial addition: {mismatches}")
print()
if mismatches == 0:
    print("Confirmed: XOR and polynomial addition over GF(2) are identical operations.")
    print("This is why Module 09 will define polynomial addition as XOR.")

---
## Section 8 — AES constant recognition

Certain byte values appear repeatedly in AES. Learning to recognize them in
polynomial form will help when we get to AES arithmetic.

**Exercise 8.1** — Run the cell below. For each AES-related byte, write in the
comment what role it plays (use the tutorial's AES connection section for hints).

In [ ]:
# Run — identify these AES-related bytes

aes_bytes = {
    0x02: "?",   # TODO: fill in the AES role
    0x03: "?",   # TODO
    0x1B: "?",   # TODO: lower 8 bits of the AES irreducible polynomial
    0x63: "?",   # TODO: AES S-box output for 0x00 input (after affine step)
    0x57: "used in AES documentation as a worked example",
    0x83: "used in AES documentation as a worked example",
}

print(f"{'Hex':<8} {'Polynomial':<40} {'AES role'}")
print("-" * 80)
for byte_val, role in aes_bytes.items():
    print(f"0x{byte_val:02X}     {byte_to_poly(byte_val):<40} {role}")

print()
print("Note: 0x1B = x^4 + x^3 + x + 1 is the reduction polynomial mod x^8")
print("The full AES irreducible polynomial is x^8 + x^4 + x^3 + x + 1 (= 0x11B).")
print("Since we work with 8-bit bytes, the x^8 term is handled implicitly.")

---
## Section 9 — Challenge: degree and term count

**Exercise 9.1** — Implement two analysis functions:

1. `poly_degree(b)` — returns the degree of the polynomial (the highest set bit position).
   Return -1 if `b == 0` (the zero polynomial has no degree by convention).

2. `term_count(b)` — returns the number of terms in the polynomial (number of 1 bits).

Then find which non-zero bytes have the **most terms** and which have the **fewest terms**.

In [ ]:
def poly_degree(b):
    """Return the degree of the polynomial represented by byte b.
    Returns -1 for the zero polynomial."""
    # TODO: implement
    pass


def term_count(b):
    """Return the number of terms (1 bits) in the polynomial."""
    # TODO: implement
    # Hint: bin(b).count('1')  is the shortest way
    pass


# Spot checks
print("Degree checks:")
print(f"  poly_degree(0x57) = {poly_degree(0x57)}  (expected 6)")
print(f"  poly_degree(0x83) = {poly_degree(0x83)}  (expected 7)")
print(f"  poly_degree(0x01) = {poly_degree(0x01)}  (expected 0)")
print(f"  poly_degree(0x00) = {poly_degree(0x00)}  (expected -1)")
print()
print("Term count checks:")
print(f"  term_count(0x57)  = {term_count(0x57)}  (expected 5)")
print(f"  term_count(0xFF)  = {term_count(0xFF)}  (expected 8)")
print(f"  term_count(0x00)  = {term_count(0x00)}  (expected 0)")
print()

# Analysis across all 256 values
max_terms = max(term_count(b) for b in range(256))
min_terms_nonzero = min(term_count(b) for b in range(1, 256))

most = [b for b in range(256) if term_count(b) == max_terms]
fewest = [b for b in range(1, 256) if term_count(b) == min_terms_nonzero]

print(f"Most terms ({max_terms}):  {[f'0x{b:02X}' for b in most]}")
print(f"Fewest terms for nonzero bytes ({min_terms_nonzero}):  first 8 = {[f'0x{b:02X}' for b in fewest[:8]]}")
print()
print("Reflection: which type of byte (most or fewest terms) is 'simpler' as a polynomial?")

---
## Section 10 — Summary and bridge to Module 09

**What you built in this notebook:**

| Function | What it does |
|---|---|
| `active_positions(b)` | Lists the bit positions where b has a 1 — these become the polynomial terms |
| `byte_to_poly(b)` | Converts a byte to its polynomial string |
| `poly_to_byte(positions)` | Converts a list of active positions back to a byte integer |
| `poly_degree(b)` | Returns the highest-degree term |
| `term_count(b)` | Counts the number of terms |

**The key insight confirmed computationally:**

> XOR on bytes is identical to term-wise addition modulo 2 on their polynomials.
> All 65,536 pairs verified.

**What Module 09 adds:**

Module 09 formalizes polynomial addition over GF(2) — the same XOR you just verified,
but with the algebraic rules that make it a proper field operation.
After that, Module 10 introduces polynomial multiplication,
which requires a reduction step analogous to the modular reduction you saw in Modules 04–07.
The byte `0x1B` you recognized in Section 8 will reappear there as a central constant.

In [ ]:
# Final summary — run to see your completed function library

print("=" * 60)
print("Module 08 Notebook — Function Summary")
print("=" * 60)

showcase = [0x00, 0x01, 0x02, 0x1B, 0x57, 0x83, 0xA5, 0xFF]
print(f"{'Hex':<7} {'Degree':<9} {'Terms':<8} {'Polynomial'}")
print("-" * 65)
for b in showcase:
    d = poly_degree(b)
    t = term_count(b)
    p = byte_to_poly(b)
    deg_str = str(d) if d >= 0 else "—"
    print(f"0x{b:02X}    {deg_str:<9} {t:<8} {p}")